In [ ]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://api.open-meteo.com/v1/forecast"
params = {
	"latitude": -21.9,
	"longitude": -46.7,
	"hourly": ["pressure_msl", "precipitation"],
	"timezone": "America/Sao_Paulo",
	"past_days": 60,
    "forecast_days": 1
}
responses = openmeteo.weather_api(url, params = params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone: {response.Timezone()}{response.TimezoneAbbreviation()}")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_pressure_msl = hourly.Variables(0).ValuesAsNumpy()
hourly_precipitation = hourly.Variables(1).ValuesAsNumpy()

hourly_data = {
	"date": pd.date_range(
		start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
		end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = hourly.Interval()),
		inclusive = "left"
	).tz_convert(response.Timezone().decode())
}

hourly_data["pressure_msl"] = hourly_pressure_msl
hourly_data["precipitation"] = hourly_precipitation

hourly_dataframe = pd.DataFrame(data = hourly_data)
print("\nHourly data\n", hourly_dataframe)




Coordinates: -21.898067474365234°N -46.711212158203125°E
Elevation: 988.0 m asl
Timezone: b'America/Sao_Paulo'b'GMT-3'
Timezone difference to GMT+0: -10800s

Hourly data
                           date  pressure_msl  precipitation
0    2026-07-24 00:00:00-03:00   1018.099976            0.0
1    2026-07-24 01:00:00-03:00   1017.799988            0.0
2    2026-07-24 02:00:00-03:00   1017.400024            0.0
3    2026-07-24 03:00:00-03:00   1017.799988            0.0
4    2026-07-24 04:00:00-03:00   1017.299988            0.1
...                        ...           ...            ...
1459 2026-09-22 19:00:00-03:00   1015.200012            0.8
1460 2026-09-22 20:00:00-03:00   1015.900024            0.7
1461 2026-09-22 21:00:00-03:00   1016.200012            0.1
1462 2026-09-22 22:00:00-03:00   1016.700012            0.4
1463 2026-09-22 23:00:00-03:00   1016.599976            0.2

[1464 rows x 3 columns]


In [24]:
import openmeteo_requests
import pandas as pd
import requests_cache
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

url = "https://api.open-meteo.com/v1/forecast"
params = {
    "latitude": -21.9,
    "longitude": -46.7,
    "hourly": ["pressure_msl", "precipitation"],
    "timezone": "America/Sao_Paulo",
    "past_days": 60,
    "forecast_days": 1
}
responses = openmeteo.weather_api(url, params = params)
response = responses[0]

hourly = response.Hourly()
hourly_pressure_msl = hourly.Variables(0).ValuesAsNumpy()
hourly_precipitation = hourly.Variables(1).ValuesAsNumpy()

hourly_data = {
    "date": pd.date_range(
        start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
        end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
        freq = pd.Timedelta(seconds = hourly.Interval()),
        inclusive = "left"
    ).tz_convert(response.Timezone().decode())
}

hourly_data["pressure_msl"] = hourly_pressure_msl
hourly_data["precipitation"] = hourly_precipitation
hourly_dataframe = pd.DataFrame(data = hourly_data)

# --- PROCESSAMENTO DOS DADOS (PRESSÃO E DIAS SEM CHUVA) ---

# 1. Pegar a data e hora exata de agora, respeitando o fuso horário da requisição
agora = pd.Timestamp.now(tz=response.Timezone().decode())

# 2. Filtrar o DataFrame para ignorar as horas futuras, pegando só do passado até agora
df_passado = hourly_dataframe[hourly_dataframe['date'] <= agora].copy()

# 3. Extrair a pressão da hora anterior mais próxima (que será a última linha do DF filtrado)
registro_atual = df_passado.iloc[-1]
pressao_atual = registro_atual['pressure_msl']

# 4. Calcular dias sem chuva agrupando a precipitação por dia
# Extraímos só a 'data' (sem as horas) e somamos a chuva de todas as 24h daquele dia
precipitacao_diaria = df_passado.groupby(df_passado['date'].dt.date)['precipitation'].sum()

dias_sem_chuva = 0
# Percorremos os dias de trás para frente (de hoje voltando pro passado)
for chuva_do_dia in precipitacao_diaria.values[::-1]:
    if chuva_do_dia > 0.2:
        break # Atingiu o limiar de chuva, interrompe a contagem
    dias_sem_chuva += 1

# --- RESULTADO FINAL ---
print(f"Data/Hora de referência : {registro_atual['date'].strftime('%d/%m/%Y %H:%M')}")
print(f"Pressão MSL atual       : {pressao_atual:.1f} hPa")
print(f"Dias sem chuva (>0.2mm) : {dias_sem_chuva}")

Data/Hora de referência : 22/09/2026 18:00
Pressão MSL atual       : 1014.5 hPa
Dias sem chuva (>0.2mm) : 0
